# Misinformation Degradation — Full Pipeline Exploration

This notebook covers:
1. Dataset statistics & structure
2. Graph-level analysis (size, depth, label distribution)
3. Drift feature analysis (cosine distance, L2 distance per hop)
4. Text divergence patterns (fake vs real)
5. Feature visualization (PCA/t-SNE of node embeddings)

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from data.upfd.loader import UPFDGraphDataset, collate_fn
from features.drift import compute_drift
from models.degradation_gnn import _build_parent_index
from torch_geometric.utils import add_self_loops, coalesce

## 1. Load Dataset

In [ ]:
SUBSET = 'gossipcop'
FEATURE_TYPE = 'bert'

train_ds = UPFDGraphDataset(root='../data/upfd/raw', subset=SUBSET, feature_type=FEATURE_TYPE, split='train')
val_ds = UPFDGraphDataset(root='../data/upfd/raw', subset=SUBSET, feature_type=FEATURE_TYPE, split='val')
test_ds = UPFDGraphDataset(root='../data/upfd/raw', subset=SUBSET, feature_type=FEATURE_TYPE, split='test')

print(f"Train: {len(train_ds)} graphs")
print(f"Val:   {len(val_ds)} graphs")
print(f"Test:  {len(test_ds)} graphs")
print(f"Total: {len(train_ds) + len(val_ds) + len(test_ds)} graphs")
print(f"Feature dim: {train_ds[0].x.shape[1]}")

## 2. Label Distribution

In [ ]:
for name, ds in [('Train', train_ds), ('Val', val_ds), ('Test', test_ds)]:
    labels = np.array([g.y.item() for g in ds])
    fake_pct = labels.mean() * 100
    print(f"{name}: {len(ds)} graphs | Fake: {labels.sum()} ({fake_pct:.1f}%) | Real: {len(ds)-labels.sum()} ({100-fake_pct:.1f}%)")

## 3. Graph Structure Statistics

In [ ]:
sizes = [g.x.size(0) for g in train_ds]
max_hops = [g.hop_depth.max().item() for g in train_ds]
edge_counts = [g.edge_index.size(1) for g in train_ds]

print(f"Node count per graph:")
print(f"  Min: {min(sizes)}, Max: {max(sizes)}, Mean: {np.mean(sizes):.1f}, Median: {np.median(sizes):.1f}")
print(f"Max hop depth:")
print(f"  Min: {min(max_hops)}, Max: {max(max_hops)}, Mean: {np.mean(max_hops):.1f}")
print(f"Edge count per graph:")
print(f"  Min: {min(edge_counts)}, Max: {max(edge_counts)}, Mean: {np.mean(edge_counts):.1f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(sizes, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Node Count Distribution')
axes[0].set_xlabel('Nodes')
axes[0].set_ylabel('Frequency')

axes[1].hist(max_hops, bins=range(0, max(max_hops)+2), edgecolor='black', alpha=0.7)
axes[1].set_title('Max Hop Depth Distribution')
axes[1].set_xlabel('Hop Depth')

fake_sizes = [g.x.size(0) for g in train_ds if g.y.item() == 1]
real_sizes = [g.x.size(0) for g in train_ds if g.y.item() == 0]
axes[2].hist(real_sizes, bins=40, alpha=0.6, label='Real', density=True)
axes[2].hist(fake_sizes, bins=40, alpha=0.6, label='Fake', density=True)
axes[2].set_title('Graph Size: Fake vs Real')
axes[2].set_xlabel('Nodes')
axes[2].legend()

plt.tight_layout()
plt.show()

## 4. Drift Feature Analysis
Compute drift features per node and compare fake vs real propagation patterns.

In [ ]:
drift_by_hop_fake = defaultdict(list)
drift_by_hop_real = defaultdict(list)

for g in train_ds:
    parent = _build_parent_index(g.edge_index, g.x.size(0))
    n_children = torch.zeros(g.x.size(0), dtype=torch.long)
    for u, v in zip(*g.edge_index[:, g.edge_index[0] != g.edge_index[1]]):
        n_children[u] += 1
    drift = compute_drift(g.x, parent, g.hop_depth, n_children)
    label = g.y.item()
    for i in range(g.x.size(0)):
        hop = g.hop_depth[i].item()
        if hop == 0:
            continue
        d = drift[i].numpy()
        target = drift_by_hop_fake if label == 1 else drift_by_hop_real
        target[hop].append(d)

print(f"Fake graphs: {len([g for g in train_ds if g.y.item()==1])}")
print(f"Real graphs: {len([g for g in train_ds if g.y.item()==0])}")

In [ ]:
drift_names = ['Cosine Dist', 'L2 Dist', 'Depth', 'N Children', 'Is Root']
max_hop = min(max(drift_by_hop_fake.keys(), default=0), max(drift_by_hop_real.keys(), default=0), 15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cosine distance over hops
hop_range = list(range(1, max_hop + 1))
fake_cosine = [np.mean([d[0] for d in drift_by_hop_fake.get(h, [0])]) if drift_by_hop_fake.get(h) else np.nan for h in hop_range]
real_cosine = [np.mean([d[0] for d in drift_by_hop_real.get(h, [0])]) if drift_by_hop_real.get(h) else np.nan for h in hop_range]

axes[0].plot(hop_range, fake_cosine, 'r-o', label='Fake', linewidth=2)
axes[0].plot(hop_range, real_cosine, 'b-s', label='Real', linewidth=2)
axes[0].set_title('Cosine Distance to Parent by Hop')
axes[0].set_xlabel('Hop Depth')
axes[0].set_ylabel('Cosine Distance (1 - sim)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# L2 distance over hops
fake_l2 = [np.mean([d[1] for d in drift_by_hop_fake.get(h, [0])]) if drift_by_hop_fake.get(h) else np.nan for h in hop_range]
real_l2 = [np.mean([d[1] for d in drift_by_hop_real.get(h, [0])]) if drift_by_hop_real.get(h) else np.nan for h in hop_range]

axes[1].plot(hop_range, fake_l2, 'r-o', label='Fake', linewidth=2)
axes[1].plot(hop_range, real_l2, 'b-s', label='Real', linewidth=2)
axes[1].set_title('L2 Distance to Parent by Hop')
axes[1].set_xlabel('Hop Depth')
axes[1].set_ylabel('L2 Distance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Node Embedding Visualization (PCA)

In [ ]:
from sklearn.decomposition import PCA

all_feats = []
all_labels = []
all_hops = []

for g in train_ds:
    all_feats.append(g.x.numpy())
    all_labels.extend([g.y.item()] * g.x.size(0))
    all_hops.extend(g.hop_depth.numpy().tolist())

all_feats = np.concatenate(all_feats, axis=0)
all_labels = np.array(all_labels)
all_hops = np.array(all_hops)

print(f"Total nodes: {len(all_feats)}")
print(f"Feature dim: {all_feats.shape[1]}")

# Subsample for visualization
n_vis = min(5000, len(all_feats))
idx = np.random.choice(len(all_feats), n_vis, replace=False)

pca = PCA(n_components=2)
emb_2d = pca.fit_transform(all_feats[idx])
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

labels_vis = all_labels[idx]
hops_vis = all_hops[idx]

axes[0].scatter(emb_2d[labels_vis==0, 0], emb_2d[labels_vis==0, 1],
                alpha=0.3, s=5, c='blue', label='Real')
axes[0].scatter(emb_2d[labels_vis==1, 0], emb_2d[labels_vis==1, 1],
                alpha=0.3, s=5, c='red', label='Fake')
axes[0].set_title('Node Embeddings by Label')
axes[0].legend()

sc = axes[1].scatter(emb_2d[:, 0], emb_2d[:, 1],
                     alpha=0.3, s=5, c=hops_vis, cmap='viridis')
axes[1].set_title('Node Embeddings by Hop Depth')
plt.colorbar(sc, ax=axes[1], label='Hop Depth')

plt.tight_layout()
plt.show()

## 6. Propagation Tree Examples

In [ ]:
# Show stats for a few example graphs
for i in [0, len(train_ds)//4, len(train_ds)//2]:
    g = train_ds[i]
    label = 'FAKE' if g.y.item() == 1 else 'REAL'
    print(f"Graph {i} ({label}): {g.x.size(0)} nodes, {g.edge_index.size(1)} edges, max hop={g.hop_depth.max().item()}")
    hops, counts = g.hop_depth.unique(return_counts=True)
    for h, c in zip(hops.tolist(), counts.tolist()):
        print(f"  Hop {h}: {c} nodes")

## 7. Summary

Key findings:
- **Graph sizes vary widely** — some trees are deep, others shallow
- **Drift patterns differ** between fake and real news at deeper hops
- **Node embeddings show separation** by label but also overlap (challenging dataset)
- **Fake news tends to have higher text divergence** from source at deeper hops